# 05 — Run the whole pipeline

Owner: the whole group. **This is the notebook to open on Google Colab (CPU runtime).** It loads
notebooks 00–04 with `%run -i` (definitions only, `RUN_TEST = False`) and then runs:

1. **Discovery** (`01`): `discover_all()` builds or resumes `state/discovery_manifest.json`.
2. **Sources** (`02`): every source version, one connection to arXiv, `ARXIV_DELAY` apart (at least 3 s),
   only `.tex`/`.bib` kept. **References** (`03`, Semantic Scholar batch endpoint, 100 papers per request)
   run at the same time in one background thread; the two rate limits are separate.
3. **Report and benchmark**: imports `references_export.zip` if present, writes `logs/papers.csv`
   (download columns + reference counts), adds this session to `logs/benchmark_sessions.jsonl` and
   recomputes `logs/benchmark.json` over all sessions (details in section 3), writes
   `logs/performance.json` for `04_monitor` and prints the Milestone 1 statistics.

**Resume.** After a Colab disconnect, run all cells again. Discovery batches, download receipts and
`references.json` files are reused, so only unfinished work is requested again.

**Time.** About 3 s × the number of source versions (roughly a day for 20,000 IDs), so expect 2–3
Colab sessions. The benchmark adds up all of them (`logs/benchmark_sessions.jsonl`); the downloader also
keeps its own `state/download/run_history.jsonl`.

## 0. Setup

Choose which stages to run. References use the Semantic Scholar batch endpoint
(100 papers per request), so they run here in a background thread and finish long before the download.
If a teammate runs `07_run_references.ipynb` instead, set `RUN_REFERENCES = False`; the report step
imports their `references_export.zip`. Every stage resumes from saved state, so leaving all of them on is
safe after a disconnect. `TEST_VERSIONS` (set by `06_test_100_versions.ipynb`) switches to a test
run: only the first assigned IDs, up to that many source versions, written to
`test_runs/<N>_versions/` so the real crawl folders are not touched. `ARXIV_CONTACT` (optional, an email address) is added to the User-Agent.

In [ ]:
# On Colab: clone or update branch VA, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # Every run, not only the first one in this runtime: otherwise the notebooks loaded with %run
    # below stay at the commit cloned first, and fixes pushed later never arrive.
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "VA"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                        "https://github.com/thinhelzan842/intro-ds-arxiv-project.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

RUN_TEST = False        # True also runs each notebook's own tests (01 and 03 call the live APIs)
RUN_DISCOVERY = True    # 01: build / resume the discovery manifest
RUN_DOWNLOAD = True     # 02: download / resume every source version
RUN_REFERENCES = True   # 03: S2 batch requests in a background thread (False if a teammate runs 07)
TEST_VERSIONS = None    # e.g. 100: test run on the first IDs only (see 06_test_100_versions)
FLUSH_DRIVE_AT_END = True   # flush and unmount Google Drive after the report, so every file is saved
MEASURE_DISK = False        # True: walk data/ on Drive for the exact disk usage (about an hour at 27,000
                            # versions); False: from the receipts, in seconds (see section 3)

# Another notebook (06) can set PIPELINE_OVERRIDES = {...} before %run -i to change the values above.
globals().update(globals().get("PIPELINE_OVERRIDES", {}))

In [ ]:
get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
if TEST_VERSIONS:
    # Test run: move every folder before 01-04 derive their paths from them.
    TEST_ROOT = PROJECT_ROOT / "test_runs" / f"{TEST_VERSIONS}_versions"
    DATA_DIR, LOG_DIR, STATE_DIR = TEST_ROOT / "data" / "raw", TEST_ROOT / "logs", TEST_ROOT / "state"
    PAPER_LOG = LOG_DIR / "papers.csv"
    for folder in (DATA_DIR, LOG_DIR, STATE_DIR):
        folder.mkdir(parents=True, exist_ok=True)
    print(f"TEST RUN: first {TEST_VERSIONS:,} source versions -> {TEST_ROOT}")

for notebook in ("01_discovery", "02_download", "03_references", "04_monitor"):
    get_ipython().run_line_magic("run", f"-i ./{notebook}.ipynb")

from concurrent.futures import ThreadPoolExecutor
import threading

class PipelineMonitor(PerformanceMonitor):
    """04's PerformanceMonitor, sampling RAM only.

    Walking data/ on Drive every few minutes (what PerformanceMonitor does for disk) slows the crawl,
    so disk is measured once at the end of each session instead (record_session in section 3).
    """

    def _sample_loop(self):
        while not self._stop_event.is_set():
            try:
                mem_mb = self._process.memory_info().rss / (1024 * 1024)
                self.ram_samples.append((time.time() - self.start_wall_time, mem_mb))
                self.peak_ram_mb = max(self.peak_ram_mb, mem_mb)
            except Exception:
                pass
            self._stop_event.wait(self.sample_interval)

    def start(self):
        self.start_wall_time = time.time()
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._sample_loop, daemon=True)
        self._thread.start()

    def stop(self):
        if self.end_wall_time is None:
            self.end_wall_time = time.time()
        self._stop_event.set()
        if self._thread and self._thread.is_alive():
            self._thread.join(timeout=2.0)


SESSIONS_DIR = LOG_DIR / "sessions"                        # one record per Colab session
BENCHMARK_SESSIONS = LOG_DIR / "benchmark_sessions.jsonl"   # all sessions, rewritten by the report (read by the book)
RAM_TIMESERIES = LOG_DIR / "ram_timeseries.csv"


def load_sessions():
    """Every session record, oldest first (imports benchmark_sessions.jsonl written by older versions)."""
    if not any(SESSIONS_DIR.glob("*.json")) and BENCHMARK_SESSIONS.is_file():
        SESSIONS_DIR.mkdir(parents=True, exist_ok=True)
        lines = [line for line in BENCHMARK_SESSIONS.read_text(encoding="utf-8").splitlines() if line.strip()]
        for n, line in enumerate(lines):
            record = json.loads(line)
            record.setdefault("session", f"legacy-{n}")
            record.setdefault("complete", True)
            write_json(SESSIONS_DIR / f"{record['session']}.json", record)
    records = [read_json(path) for path in SESSIONS_DIR.glob("*.json")] if SESSIONS_DIR.is_dir() else []
    return sorted((r for r in records if r), key=lambda r: r["start"])


class SessionRecorder:
    """Saves this session's wall time and RAM every `every` seconds, so a killed session still counts.

    One file per session in logs/sessions/ (overwritten each time, `complete: false` until the report cell
    writes the final record with the disk usage); new RAM samples are appended to logs/ram_timeseries.csv.
    """

    def __init__(self, monitor, every=300):
        self.monitor, self.every = monitor, every
        started = datetime.fromtimestamp(monitor.start_wall_time, tz=timezone.utc)
        self.id = started.strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
        self.path = SESSIONS_DIR / f"{self.id}.json"
        # Until this session measures its own disk usage, count what the earlier sessions left.
        self.base_bytes = max((s.get("project_bytes", 0) for s in load_sessions()), default=0)
        self.flushed = 0
        self._guard = threading.Lock()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def _loop(self):
        while not self._stop.wait(self.every):
            try:
                self.save()
            except Exception as err:   # a full Drive must not kill the crawl
                print(f"[benchmark] could not save this session: {err!r}")

    def save(self, final=None):
        with self._guard:
            monitor = self.monitor
            end = monitor.end_wall_time or time.time()
            samples = list(monitor.ram_samples)
            rams = [mb for _, mb in samples]
            record = {
                "session": self.id, "complete": final is not None,
                "start": to_iso(datetime.fromtimestamp(monitor.start_wall_time, tz=timezone.utc)),
                "end": to_iso(datetime.fromtimestamp(end, tz=timezone.utc)),
                "wall_seconds": round(end - monitor.start_wall_time, 3),
                "stages_seconds": dict(monitor.stages),
                "ram_peak_mb": round(max(rams, default=0.0), 2),
                "ram_avg_mb": round(sum(rams) / len(rams), 2) if rams else None,
                "ram_samples": len(rams),
                "local_temp_peak_bytes": 0, "project_bytes": self.base_bytes, "output_bytes": 0,
            }
            record.update(final or {})
            SESSIONS_DIR.mkdir(parents=True, exist_ok=True)
            write_json(self.path, record)
            new = samples[self.flushed:]
            if new:
                header = not RAM_TIMESERIES.exists()
                with RAM_TIMESERIES.open("a", newline="", encoding="utf-8") as stream:
                    writer = csv.writer(stream)
                    if header:
                        writer.writerow(["utc_time", "rss_mb"])
                    for seconds, mb in new:
                        moment = datetime.fromtimestamp(monitor.start_wall_time + seconds, tz=timezone.utc)
                        writer.writerow([to_iso(moment), round(mb, 2)])
                self.flushed = len(samples)
            return record

    def finish(self, final):
        self._stop.set()
        self._thread.join(timeout=5)
        return self.save(final=final)


# One pipeline at a time per Drive (05 and the 06 test included): a second runtime stops here.
PIPELINE_LOCK = RunLock(PROJECT_ROOT / "pipeline.lock").acquire()

monitor = PipelineMonitor(sample_interval=5, data_dir=DATA_DIR)   # RAM every 5 s
monitor.start()
recorder = SessionRecorder(monitor)                                 # saved every 5 minutes

## 1. Discovery

A **Discovery** bar shows the IDs checked out of all assigned IDs, with the papers and versions found.

Skipped quickly when the manifest is already complete. `discovery_seconds` adds up the API time of every
session, so the report shows the real discovery time and not just this session's.

In [ ]:
def discover_first_versions(target_versions):
    """Test run: discover assigned IDs in order until they hold at least target_versions versions."""
    ids = all_assigned_ids()
    n = max(len((read_json(MANIFEST) or {"papers": []})["papers"]), ARXIV_BATCH_SIZE)
    any_query = False
    while True:
        found_papers, queried, seconds = discover_all(ids[:n])
        any_query = any_query or queried
        if sum(p["versions"] for p in found_papers) >= target_versions or n >= len(ids):
            break
        n += ARXIV_BATCH_SIZE
    sample, total = [], 0
    for paper in found_papers:
        if total >= target_versions:
            break
        sample.append(paper)
        total += paper["versions"]
    return sample, any_query, seconds


did_query = False
if TEST_VERSIONS:
    monitor.start_stage("discovery")
    papers, did_query, discovery_seconds = discover_first_versions(TEST_VERSIONS)
    monitor.end_stage("discovery")
    validate_manifest(papers)
    DOWNLOAD_MANIFEST = STATE_DIR / "test_manifest.json"
    write_json(DOWNLOAD_MANIFEST, {"papers": papers})
elif RUN_DISCOVERY:
    monitor.start_stage("discovery")
    papers, did_query, discovery_seconds = discover_all()
    monitor.end_stage("discovery")
else:
    saved = read_json(MANIFEST)
    if saved is None:
        raise FileNotFoundError(f"{MANIFEST} is missing: run with RUN_DISCOVERY = True first")
    papers, discovery_seconds = saved["papers"], saved.get("discovery_seconds", 0.0)

if not TEST_VERSIONS:
    validate_assigned_manifest(papers)
    DOWNLOAD_MANIFEST = MANIFEST
if did_query:
    time.sleep(ARXIV_DELAY)   # keep the API-to-source transition at least 3 s apart
found = [p["id"] for p in papers if p.get("versions", 0) > 0]   # IDs that exist on arXiv
print(f"{len(papers):,} IDs, {len(found):,} found on arXiv, "
      f"{sum(p.get('versions', 0) for p in papers):,} source versions")

## 2. Sources and references

`download_sources` blocks this cell for the whole crawl. Two progress bars: **Papers** (papers done out of
every discovered paper, with versions, MB downloaded and the ETA) and the **file being downloaded** (its
size and the MB left). A text line every 100 papers stays in the output as a log.
Stopping the cell (or a disconnect) is safe: archives already fetched are processed first, and the
references thread stops after its current batch request.

In [ ]:
stop_references = threading.Event()


def fetch_missing_references(ids, stop):
    """References of the unfinished papers (S2 batch endpoint, S2_BATCH_SIZE per request); stops once `stop` is set.

    The unfinished papers are found once, from logs/references.csv (pending_references), so a restart
    does not read 20,000 references.json files on Drive.
    `stop` is passed in, not read from the global: a rerun rebinds stop_references, and a thread left
    by an interrupted run must keep seeing its own (set) event instead of the new one.
    """
    todo = pending_references(ids)
    print(f"[references] {len(todo):,} of {len(ids):,} papers still need references")
    fetched = 0
    for start in range(0, len(todo), S2_BATCH_SIZE):
        if stop.is_set():
            break
        fetched += len(fetch_references_batch(todo[start:start + S2_BATCH_SIZE]))
        if (start // S2_BATCH_SIZE + 1) % 20 == 0:
            print(f"[references] {min(start + S2_BATCH_SIZE, len(todo)):,}/{len(todo):,} papers requested")
    return fetched


# A second interrupt can cut the shutdown below short; such a thread ends after its current batch.
leftover = [t.name for t in threading.enumerate() if t.name.startswith("references") and t.is_alive()]
if leftover:
    print(f"[references] {len(leftover)} thread(s) of an interrupted run still finishing their current batch")

references_pool = references_future = download_summary = None
try:
    if RUN_REFERENCES:
        monitor.start_stage("references")
        references_pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="references")
        references_future = references_pool.submit(fetch_missing_references, found, stop_references)
        references_future.add_done_callback(lambda _: monitor.end_stage("references"))
    if RUN_DOWNLOAD:
        monitor.start_stage("download")
        download_summary = download_sources(DOWNLOAD_MANIFEST, DATA_DIR, DOWNLOAD_STATE, user_agent=DOWNLOAD_USER_AGENT,
                                            workers=DOWNLOAD_WORKERS, verify=DOWNLOAD_VERIFY, progress=True)
        monitor.end_stage("download")
        if download_summary["interrupted"] or download_summary.get("stopped_reason"):
            stop_references.set()
    if references_future is not None:
        if not stop_references.is_set():
            print("[references] download finished; waiting for the remaining references ...")
        print(f"[references] {references_future.result():,} papers fetched in this session "
              f"({REFERENCE_STATS['requests']:,} requests, {REFERENCE_STATS['rate_limited']:,} HTTP 429)")
except KeyboardInterrupt:
    print("Interrupted; saved work is kept and the next run resumes from it.")
finally:
    stop_references.set()
    if references_pool is not None:
        references_pool.shutdown(wait=True)

## 3. Paper log and benchmark

If `references_export.zip` from `07_run_references.ipynb` is in the project folder (next to `data/`),
its `references.json` files and `references.csv` are imported first. A saved `ok` file is never
replaced by an `error` one, so importing the same or a newer zip again is safe.

`logs/papers.csv` (`LOG_FIELDS` from `00_config`) is rebuilt from the downloader's `state/download/papers.csv`
plus the reference counts in `logs/references.csv` (`references_status` in `03`: per paper its last `ok` row,
else its last row; `references.json` is read only for papers the log does not show as finished).

**Benchmark over every session.** The crawl takes 2–3 Colab sessions. Each session saves its record (wall
time, RAM) to `logs/sessions/` every 5 minutes, so a session that Colab kills still counts, and appends its
RAM samples to `logs/ram_timeseries.csv`; this cell adds the disk usage and rewrites
`logs/benchmark_sessions.jsonl` from all of them.
`logs/benchmark.json` is then recomputed from all sessions, with the lab book's definitions:

| Lab book | Here |
|---|---|
| papers scraped successfully, success rate | papers whose every version was downloaded and processed (LaTeX or PDF-only); rate over the assigned IDs and over the papers that exist on arXiv |
| average size before / after removing figures | per paper, summed over its versions; papers with at least one LaTeX version |
| references per paper, reference success rate | per paper from `logs/references.csv` (last `ok` row, else last row, the rule `references.json` follows) |
| time per paper, total time, discovery time | end-to-end wall time of all sessions / papers on arXiv; per-paper download time (median); discovery API time of all sessions |
| maximum / average RAM | max over sessions (5 s samples and the downloader's 2 s samples); time-weighted average |
| maximum disk, final output size | `data/` + `state/` + `logs/` at each session end, plus the downloader's peak of raw archives in `/tmp`; `data/raw` alone. With `MEASURE_DISK = True` both are `du` walks (about an hour on Drive at 27,000 versions); by default `data/raw` is the `.tex`/`.bib` bytes in the receipts plus one `references.json` per paper at the average size of 50 of them, and `state/download/receipts/` (one file per version) is counted as the size of `receipts.jsonl` |

`logs/performance.json` is written in `04_monitor`'s format from the same numbers (with `ram_timeseries`), so
its plots and the Jupyter Book use them. The four charts of `04_monitor` (RAM over time, size before/after,
versions per paper, references per paper) are saved as PNG in **`logs/figures/`**, redrawn from all data
every run. A session killed before this cell counts up to its last
save (at most 5 minutes lost), without its disk usage.

At the end the pipeline lock is released and Google Drive is flushed and unmounted (`FLUSH_DRIVE_AT_END`).

In [ ]:
import statistics

REFERENCES_EXPORT = PROJECT_ROOT / "references_export.zip"   # written by 07_run_references.ipynb
_REFERENCE_MEMBER = re.compile(r"^data/raw/([A-Za-z0-9._-]+)/references\.json$")
BENCHMARK_FILE = LOG_DIR / "benchmark.json"
MB = 1024 * 1024


def import_references(zip_path):
    """Copy references.json files and references.csv from a 07 export into DATA_DIR / LOG_DIR."""
    imported = kept = 0
    with zipfile.ZipFile(zip_path) as package:
        for name in package.namelist():
            match = _REFERENCE_MEMBER.match(name)
            if not match:
                continue
            doc = json.loads(package.read(name))
            target = DATA_DIR / match.group(1) / "references.json"
            previous = read_json(target)
            if previous and previous.get("status") == "ok" and doc.get("status") != "ok":
                kept += 1   # never replace a good result with an error
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            write_json(target, doc)
            imported += 1
        if "logs/references.csv" in package.namelist():
            log_bytes = package.read("logs/references.csv")
            (LOG_DIR / "references.csv").write_bytes(log_bytes)   # the 07 log replaces any older copy
    return imported, kept


def write_paper_log(refs, download_csv=None, path=None):
    """logs/papers.csv for 04_monitor: download columns from 02 + reference counts from 03."""
    download_csv = Path(download_csv or DOWNLOAD_STATE / "papers.csv")
    path = Path(path or PAPER_LOG)
    with download_csv.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    for row in rows:
        doc = refs.get(row["id"])
        if doc and doc.get("status") == "ok":
            row["n_refs"], row["n_arxiv_refs"] = doc["n_refs"], doc["n_arxiv_refs"]
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=LOG_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)
    return len(rows)


def du_bytes(*folders, skip=()):
    """Bytes of all files under the folders, like `du -sb` (one os.scandir walk), leaving out the skip folders."""
    skip = {Path(folder) for folder in skip}
    total, stack = 0, [Path(f) for f in folders if Path(f).exists()]
    while stack:
        with os.scandir(stack.pop()) as entries:
            for entry in entries:
                try:
                    if entry.is_dir(follow_symlinks=False):
                        if Path(entry.path) not in skip:
                            stack.append(Path(entry.path))
                    elif entry.is_file(follow_symlinks=False):
                        total += entry.stat(follow_symlinks=False).st_size
                except OSError:
                    pass
    return total


def estimated_disk_bytes(refs):
    """(project_bytes, output_bytes) without walking data/ on Drive (MEASURE_DISK = False).

    data/raw: the .tex/.bib bytes in the receipts, plus one references.json per paper with a result at the
    average size of up to 50 of them. state/ and logs/ are walked, except state/download/receipts/ (one
    small file per version, the same receipts as receipts.jsonl, whose size stands in for it).
    """
    receipts = _read_receipt_index(DOWNLOAD_STATE)
    with_refs = [pid for pid, doc in refs.items() if doc]
    sizes = [path.stat().st_size for path in map(references_path, with_refs[:50]) if path.is_file()]
    output = sum(r["after_bytes"] for r in receipts.values())
    if sizes:
        output += len(with_refs) * sum(sizes) // len(sizes)
    index = DOWNLOAD_STATE / "receipts.jsonl"
    state_and_logs = du_bytes(STATE_DIR, LOG_DIR, skip=[DOWNLOAD_STATE / "receipts"])
    return output + state_and_logs + (index.stat().st_size if index.is_file() else 0), output


def record_session(recorder, download_summary, refs):
    """Final record of this session (with disk usage) and benchmark_sessions.jsonl of all sessions."""
    monitor = recorder.monitor
    monitor.stop()
    rams = [mb for _, mb in monitor.ram_samples]
    download_summary = download_summary or {}
    downloader_peak_mb = (download_summary.get("ram_peak_bytes") or 0) / MB   # sampled every 2 s
    if MEASURE_DISK:
        output_bytes = du_bytes(DATA_DIR)   # walked once: data/ is the slow part on Drive
        project_bytes = output_bytes + du_bytes(STATE_DIR, LOG_DIR)
    else:
        project_bytes, output_bytes = estimated_disk_bytes(refs)
    session = recorder.finish({
        "ram_peak_mb": round(max([downloader_peak_mb, *rams]), 2),
        "local_temp_peak_bytes": download_summary.get("local_temp_peak_bytes", 0),
        "project_bytes": project_bytes,
        "output_bytes": output_bytes,
    })
    BENCHMARK_SESSIONS.write_text("".join(json.dumps(s) + "\n" for s in load_sessions()), encoding="utf-8")
    return session


def compute_benchmark(papers, refs, discovery_seconds):
    """Milestone 1 benchmark over every session so far (definitions in the table above)."""
    sessions = load_sessions()
    records = validate_manifest(papers)
    found = [r for r in records if r["version_count"] > 0]
    receipts = _read_receipt_index(DOWNLOAD_STATE)
    per_paper = []
    for record in found:
        got = [receipts.get((record["arxiv_id"], v)) for v in range(1, record["version_count"] + 1)]
        per_paper.append({
            "done": all(got),
            "latex": any(g and g["status"] == "ok" for g in got),
            "before": sum(g["before_bytes"] for g in got if g),
            "after": sum(g["after_bytes"] for g in got if g),
            "seconds": sum(g.get("elapsed_seconds", 0) for g in got if g),
        })
    done = [p for p in per_paper if p["done"]]
    sized = [p for p in done if p["latex"]]
    docs = [refs.get(r["arxiv_id"]) or {} for r in found]
    ok_docs = [d for d in docs if d.get("status") == "ok"]
    wall = sum(s["wall_seconds"] for s in sessions)

    def share(part, whole):
        return round(100 * part / whole, 2) if whole else None

    def mean(values):
        return round(sum(values) / len(values), 2) if values else None

    before, after = mean([p["before"] for p in sized]), mean([p["after"] for p in sized])
    return {
        "computed_at": to_iso(datetime.now(timezone.utc)),
        "sessions": len(sessions),
        "ids_assigned": len(records),
        "papers_on_arxiv": len(found),
        "versions_total": sum(r["version_count"] for r in found),
        "versions_processed": sum(1 for r in found for v in range(1, r["version_count"] + 1)
                                  if (r["arxiv_id"], v) in receipts),
        "papers_scraped": len(done),
        "papers_with_latex": len(sized),
        "success_rate_of_assigned_pct": share(len(done), len(records)),
        "success_rate_of_papers_on_arxiv_pct": share(len(done), len(found)),
        "avg_paper_bytes_before": before,
        "avg_paper_bytes_after": after,
        "space_saved_pct": share(before - after, before) if before else None,
        "references_ok": len(ok_docs),
        "references_not_found": sum(d.get("status") == "not_found" for d in docs),
        "references_error_or_missing": sum(d.get("status") not in ("ok", "not_found") for d in docs),
        "reference_success_rate_pct": share(len(ok_docs), len(found)),
        "avg_references_per_paper": mean([d["n_refs"] for d in ok_docs]),
        "avg_arxiv_references_per_paper": mean([d["n_arxiv_refs"] for d in ok_docs]),
        "discovery_seconds": round(discovery_seconds, 2),
        "wall_seconds_total": round(wall, 2),
        "avg_seconds_per_paper_end_to_end": round(wall / len(found), 3) if found else None,
        "paper_download_seconds_median": round(statistics.median(p["seconds"] for p in done), 3) if done else None,
        "ram_peak_mb": max((s["ram_peak_mb"] for s in sessions), default=None),
        "ram_avg_mb": round(sum((s["ram_avg_mb"] or 0) * s["wall_seconds"] for s in sessions) / wall, 2) if wall else None,
        "max_disk_bytes": max((s["project_bytes"] + s["local_temp_peak_bytes"] for s in sessions), default=0),
        "final_output_bytes": sessions[-1]["output_bytes"] if sessions else 0,
    }


FIGURES_DIR = LOG_DIR / "figures"


def save_figures(performance, paper_log=None):
    """04_monitor's charts as PNG in logs/figures/ (also shown in the notebook); returns the saved names."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    papers_df = load_papers_dataframe(paper_log or PAPER_LOG)
    papers_df = papers_df[papers_df["versions"] > 0]   # IDs not on arXiv are not papers (no "v0" bar)
    charts = {
        "ram_usage.png": lambda path: plot_ram_usage(performance["ram_timeseries"], performance["peak_ram_mb"],
                                                     performance["avg_ram_mb"], save_path=path),
        "size_comparison.png": lambda path: plot_size_comparison(papers_df, save_path=path),
        "versions_distribution.png": lambda path: plot_versions_distribution(papers_df, save_path=path),
        "references_distribution.png": lambda path: plot_references_distribution(papers_df, save_path=path),
    }
    saved = []
    for name, draw in charts.items():
        path = FIGURES_DIR / name
        path.unlink(missing_ok=True)        # never keep a chart drawn from older data
        try:
            draw(path)
        except Exception as err:            # a chart must never stop the pipeline
            print(f"Could not draw {name}: {err!r}")
        finally:
            plt.close("all")
        if path.is_file():
            saved.append(name)
    return saved


def performance_for_monitor(bench):
    """logs/performance.json in 04_monitor's format, from all sessions, with the RAM time series."""
    sessions = load_sessions()
    series = []
    with RAM_TIMESERIES.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    if rows:
        first = datetime.fromisoformat(rows[0]["utc_time"])
        step = max(1, len(rows) // 5000)   # at most ~5,000 points for the plots
        series = [[round((datetime.fromisoformat(r["utc_time"]) - first).total_seconds(), 1), float(r["rss_mb"])]
                  for r in rows[::step]]
    stages = {"discovery": bench["discovery_seconds"]}
    for name in ("download", "references"):
        stages[name] = round(sum(s["stages_seconds"].get(name, 0) for s in sessions), 3)
    return {
        "start_time": sessions[0]["start"], "end_time": sessions[-1]["end"],
        "total_wall_time_seconds": bench["wall_seconds_total"],
        "peak_ram_mb": bench["ram_peak_mb"], "avg_ram_mb": bench["ram_avg_mb"],
        "initial_disk_mb": 0.0,
        "max_disk_mb": round(bench["max_disk_bytes"] / MB, 2),
        "final_disk_mb": round(bench["final_output_bytes"] / MB, 2),
        "samples_count": sum(s["ram_samples"] for s in sessions),
        "stages": stages,
        "ram_timeseries": series,
    }


if REFERENCES_EXPORT.is_file() and not TEST_VERSIONS:
    imported, kept = import_references(REFERENCES_EXPORT)
    print(f"Imported {imported:,} references.json files from {REFERENCES_EXPORT.name}"
          + (f" ({kept:,} older ok files kept)" if kept else ""))

refs = references_status(found)   # logs/references.csv, shared below
session = record_session(recorder, download_summary, refs)
print(f"Session: {session['wall_seconds'] / 60:.1f} min, peak RAM {session['ram_peak_mb']:.0f} MB, "
      f"project on disk {session['project_bytes'] / MB:,.1f} MB")

if (DOWNLOAD_STATE / "papers.csv").is_file():
    print(f"logs/papers.csv: {write_paper_log(refs):,} papers")
    benchmark = compute_benchmark(papers, refs, discovery_seconds)
    write_json(BENCHMARK_FILE, benchmark)
    performance = performance_for_monitor(benchmark)
    write_json(PERF_LOG, performance)

    b = benchmark
    rows = [
        ("Sessions", f"{b['sessions']}"),
        ("Papers scraped (all versions processed)", f"{b['papers_scraped']:,} of {b['papers_on_arxiv']:,} on arXiv "
                                                    f"({b['ids_assigned']:,} IDs assigned)"),
        ("Success rate: of assigned IDs / of papers on arXiv", f"{b['success_rate_of_assigned_pct']}% / {b['success_rate_of_papers_on_arxiv_pct']}%"),
        ("Versions processed", f"{b['versions_processed']:,} / {b['versions_total']:,}"),
        ("Avg paper size before -> after (.tex/.bib)", f"{(b['avg_paper_bytes_before'] or 0) / 1e3:,.1f} KB -> "
                                                       f"{(b['avg_paper_bytes_after'] or 0) / 1e3:,.1f} KB ({b['space_saved_pct']}% saved)"),
        ("References: ok / not_found / error or missing", f"{b['references_ok']:,} / {b['references_not_found']:,} / {b['references_error_or_missing']:,}"),
        ("Reference success rate (papers on arXiv)", f"{b['reference_success_rate_pct']}%"),
        ("Avg references per paper (arXiv refs)", f"{b['avg_references_per_paper']} ({b['avg_arxiv_references_per_paper']})"),
        ("Entry discovery time", f"{b['discovery_seconds']:,.1f} s"),
        ("Total wall time (all sessions)", f"{b['wall_seconds_total'] / 3600:,.2f} h ({b['wall_seconds_total'] / 60:,.1f} min)"),
        ("Avg time per paper, end to end", f"{b['avg_seconds_per_paper_end_to_end']} s"),
        ("Download time per paper (median)", f"{b['paper_download_seconds_median']} s"),
        ("Peak RAM / average RAM", f"{b['ram_peak_mb']} MB / {b['ram_avg_mb']} MB"),
        ("Max disk required / final output size", f"{b['max_disk_bytes'] / MB:,.1f} MB / {b['final_output_bytes'] / MB:,.1f} MB"),
    ]
    width = max(len(name) for name, _ in rows)
    print("\n" + "\n".join(f"{name:<{width}}  {value}" for name, value in rows))
    print(f"\nSaved {BENCHMARK_FILE.name}, {PERF_LOG.name} (04 format, with ram_timeseries), "
          f"{BENCHMARK_SESSIONS.name}, {RAM_TIMESERIES.name}")
    figures = save_figures(performance)
    print(f"Charts in {FIGURES_DIR}: {', '.join(figures) if figures else 'none (no data yet)'}")
else:
    performance = performance_for_monitor({"discovery_seconds": discovery_seconds, "wall_seconds_total": session["wall_seconds"],
                                           "ram_peak_mb": session["ram_peak_mb"], "ram_avg_mb": session["ram_avg_mb"],
                                           "max_disk_bytes": session["project_bytes"], "final_output_bytes": session["output_bytes"]})
    print("No download yet: run with RUN_DOWNLOAD = True to create state/download/papers.csv")

PIPELINE_LOCK.release()
if IN_COLAB and FLUSH_DRIVE_AT_END:
    from google.colab import drive
    drive.flush_and_unmount()   # writes to Drive are asynchronous: make sure all of them reached Drive
    print("Google Drive flushed and unmounted: every file of this session is saved. Run all mounts it again.")